# Dog Breed Classification using Transfer Learning (ResNet50)

**10 Breeds:** Beagle, Boxer, Bulldog, Dachshund, German Shepherd, Golden Retriever, Labrador Retriever, Poodle, Rottweiler, Yorkshire Terrier

**Approach:** Fine-tune a pretrained ResNet50 on ~967 images with data augmentation.

## 1. Install Dependencies

In [1]:
# Uncomment and run if packages are not installed
%pip install torch torchvision scikit-learn matplotlib pillow

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/114.5 MB 3.4 MB/s eta 0:00:34
   ---------------------------------------- 1.3/114.5 MB 3.7 MB/s eta 0:00:31
    --------------------------------------- 2.1/114.5 MB 3.4 MB/s eta 0:00:33
   - -------------------------------------- 2.9/114.5 MB 3.4 MB/s eta 0:00:34
   - -------------------------------------- 3.9/114.5 MB 3.8 MB/s eta 0:00:30
   - -------------------------------------- 4.5/114.5 MB 3.6 MB/s eta 0:00:31
   - -------------------------------------- 5.2/114.5 MB 3.6 MB/s eta 0:00:31
   -- ------------------------------------- 5.8/114.5 MB 3.6 MB/s eta 0:00:31
   -- ------------------------------------- 6.6/114.5 MB 3.5 MB/s eta 0:00:31
   -- -------------------------

## 2. Imports & Configuration

In [ ]:
import os
import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ─── Configuration ─────────────────────────────────────────
DATA_DIR = "dataset"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
TRAIN_SPLIT = 0.8
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Explore the Dataset

In [ ]:
# Count images per breed
breed_counts = {}
for breed in sorted(os.listdir(DATA_DIR)):
    breed_path = os.path.join(DATA_DIR, breed)
    if os.path.isdir(breed_path):
        count = len(os.listdir(breed_path))
        breed_counts[breed] = count
        print(f"{breed:<25s} {count} images")

print(f"\nTotal: {sum(breed_counts.values())} images across {len(breed_counts)} breeds")

In [ ]:
# Visualize sample images from each breed
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for idx, (breed, ax) in enumerate(zip(sorted(breed_counts.keys()), axes.flat)):
    breed_path = os.path.join(DATA_DIR, breed)
    img_name = os.listdir(breed_path)[0]
    img = Image.open(os.path.join(breed_path, img_name))
    ax.imshow(img)
    ax.set_title(breed.replace("_", " "), fontsize=10)
    ax.axis("off")
plt.suptitle("Sample Image from Each Breed", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Dataset distribution bar chart
plt.figure(figsize=(10, 5))
breeds = [b.replace("_", " ") for b in breed_counts.keys()]
counts = list(breed_counts.values())
plt.bar(breeds, counts, color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of Images")
plt.title("Images per Breed")
plt.tight_layout()
plt.show()

## 4. Data Transforms & DataLoaders

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Load dataset and create train/val split
full_dataset = datasets.ImageFolder(DATA_DIR)
class_names = full_dataset.classes
num_classes = len(class_names)

dataset_size = len(full_dataset)
indices = list(range(dataset_size))
np.random.shuffle(indices)
split = int(dataset_size * TRAIN_SPLIT)
train_indices, val_indices = indices[:split], indices[split:]

train_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(DATA_DIR, transform=val_transforms)

train_subset = torch.utils.data.Subset(train_dataset, train_indices)
val_subset = torch.utils.data.Subset(val_dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)

print(f"Classes ({num_classes}): {class_names}")
print(f"Train: {len(train_subset)} | Val: {len(val_subset)}")

In [ ]:
# Visualize augmented training samples
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = denormalize(images[i]).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(class_names[labels[i]], fontsize=8)
    ax.axis("off")
plt.suptitle("Augmented Training Samples", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Build the Model

In [ ]:
# Load pretrained ResNet50 and modify for our task
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4 for fine-tuning
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace classification head
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, num_classes),
)

model = model.to(DEVICE)

# Count parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 6. Train the Model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                  factor=0.5, patience=3)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

print(f"Training for {NUM_EPOCHS} epochs...\n")

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # ── Train phase ──
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # ── Validation phase ──
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc = correct / total
    scheduler.step(val_acc)

    elapsed = time.time() - start
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
          f"{elapsed:.1f}s")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

print(f"\nBest Validation Accuracy: {best_acc:.4f}")
model.load_state_dict(best_model_wts)

## 7. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, history["train_loss"], "b-o", markersize=4, label="Train Loss")
ax1.plot(epochs_range, history["val_loss"], "r-o", markersize=4, label="Val Loss")
ax1.set_title("Loss over Epochs", fontsize=13)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], "b-o", markersize=4, label="Train Acc")
ax2.plot(epochs_range, history["val_acc"], "r-o", markersize=4, label="Val Acc")
ax2.set_title("Accuracy over Epochs", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Evaluation & Classification Report

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("Classification Report")
print("=" * 70)
print(classification_report(all_labels, all_preds, target_names=class_names))

## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_title("Confusion Matrix", fontsize=14)
plt.colorbar(im, ax=ax)

tick_marks = np.arange(len(class_names))
short_names = [n.replace("_", "\n") for n in class_names]
ax.set_xticks(tick_marks)
ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=9)
ax.set_yticks(tick_marks)
ax.set_yticklabels(short_names, fontsize=9)

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black", fontsize=9)

ax.set_ylabel("True Label", fontsize=12)
ax.set_xlabel("Predicted Label", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

## 10. Save the Model

In [ ]:
model_path = os.path.join(OUTPUT_DIR, "dog_classifier.pth")
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "num_classes": num_classes,
}, model_path)
print(f"Model saved to: {model_path}")

## 11. Predict on New Images

In [ ]:
def predict_image(image_path, top_k=3):
    """Predict the breed of a dog image."""
    # Load model
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
    pred_model = models.resnet50(weights=None)
    pred_model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(pred_model.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, checkpoint["num_classes"]),
    )
    pred_model.load_state_dict(checkpoint["model_state_dict"])
    pred_model.to(DEVICE)
    pred_model.eval()

    names = checkpoint["class_names"]

    # Preprocess
    image = Image.open(image_path).convert("RGB")
    input_tensor = val_transforms(image).unsqueeze(0).to(DEVICE)

    # Predict
    with torch.no_grad():
        outputs = pred_model(input_tensor)
        probs = torch.softmax(outputs, dim=1)[0]

    top_probs, top_indices = probs.topk(top_k)

    # Display
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.imshow(image)
    ax1.set_title(f"Prediction: {names[top_indices[0]].replace('_', ' ')}", fontsize=13)
    ax1.axis("off")

    breed_labels = [names[i].replace("_", " ") for i in top_indices]
    confidences = [p.item() * 100 for p in top_probs]
    colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(top_k)]
    ax2.barh(breed_labels[::-1], confidences[::-1], color=colors[::-1])
    ax2.set_xlabel("Confidence (%)")
    ax2.set_title("Top Predictions")
    ax2.set_xlim(0, 100)

    plt.tight_layout()
    plt.show()

    for i in range(top_k):
        print(f"  {i+1}. {breed_labels[i]:<25s} {confidences[i]:5.1f}%")

In [ ]:
# Example: predict on a sample from the dataset
# Change this path to test with your own image
sample_breed = class_names[0]
sample_dir = os.path.join(DATA_DIR, sample_breed)
sample_image = os.path.join(sample_dir, os.listdir(sample_dir)[0])

print(f"Testing with: {sample_image}")
predict_image(sample_image, top_k=5)

In [ ]:
# Predict on your own image — just change the path below:
# predict_image("path/to/your/dog_photo.jpg", top_k=5)